<a href="https://colab.research.google.com/github/snehaxavier2110/Mtech_coursework/blob/main/mlp_electricity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
data=pd.read_csv('/content/lstm_forecast_comparison.csv')

In [ ]:
data

,date,152,lstm_forecast,sarima_forecast,Unnamed: 4,Unnamed: 5
0,01-01-2018 03:00,796,773.686584,820.627100,NaN,NaN
1,01-01-2018 04:00,746,749.334168,718.173882,NaN,NaN
2,01-01-2018 05:00,762,727.288696,690.631164,NaN,NaN
3,01-01-2018 06:00,802,774.834534,748.946800,NaN,NaN
4,01-01-2018 07:00,863,856.281311,847.535625,NaN,NaN
5,01-01-2018 08:00,870,933.284241,904.162003,NaN,NaN
6,01-01-2018 09:00,913,977.201782,915.990411,NaN,NaN
7,01-01-2018 10:00,1064,1120.973633,1065.967452,NaN,NaN
8,01-01-2018 11:00,1379,1257.217041,1124.009391,NaN,NaN
9,01-01-2018 12:00,1635,1389.403564,1452.134065,NaN,NaN


In [ ]:
X = data[['lstm_forecast', 'sarima_forecast']]
y = data['152']

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

model = Sequential()
model.add(Input(shape=(X.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='linear'))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,305 (9.00 KB)

 Trainable params: 2,305 (9.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X, y, epochs=100, batch_size=32)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - loss: 1225824.0000 - mae: 1056.0941
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 1139032.2500 - mae: 1017.5311
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 1055051.5000 - mae: 978.7891
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 974358.8125 - mae: 940.0759
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 905010.1875 - mae: 905.4763
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 838766.0625 - mae: 871.1291
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 775326.5000 - mae: 836.9495
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 715340.3125 - mae: 803.3245
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 658715.9375 - mae: 770.1636
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 604508.6875 - mae: 737.0283
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 552560.6875 - mae: 703.8105
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/ste

In [ ]:
from sklearn.metrics import mean_absolute_error

mlp_predictions = model.predict(X)

lstm_mae = mean_absolute_error(y, data['lstm_forecast'])
sarima_mae = mean_absolute_error(y, data['sarima_forecast'])
mlp_mae = mean_absolute_error(y, mlp_predictions)

print(f'MAE for LSTM forecast: {lstm_mae:.2f}')
print(f'MAE for SARIMA forecast: {sarima_mae:.2f}')
print(f'MAE for MLP predictions: {mlp_mae:.2f}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
MAE for LSTM forecast: 90.09
MAE for SARIMA forecast: 96.56
MAE for MLP predictions: 86.27


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

# Define the Early Stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


# Re-compile and train the model with the Early Stopping callback
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train, y_train, epochs=100, batch_size=32, callbacks=[early_stopping], validation_data=(X_val, y_val)) # Add validation_data=(X_val, y_val) if you have a validation set

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 10669.8877 - mae: 79.8069 - val_loss: 35450.9570 - val_mae: 128.7631
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step - loss: 9788.0078 - mae: 72.8917 - val_loss: 40693.0547 - val_mae: 146.9712
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - loss: 10415.1289 - mae: 73.6057 - val_loss: 38733.9141 - val_mae: 140.4358
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - loss: 10122.5557 - mae: 72.7868 - val_loss: 34342.3320 - val_mae: 126.2003
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - loss: 9724.4189 - mae: 73.2549 - val_loss: 30393.7910 - val_mae: 117.8763
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 9778.0596 - mae: 74.6668 - val_loss: 28384.5664 - val_mae: 115.0701
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - loss: 10020.9131 - mae: 75.5669 - val_loss: 28426.0508 - val_mae: 115.1368
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 10014.2393 - mae: 75.5263 - val_loss: 29990.0664 - va

In [ ]:
from sklearn.metrics import mean_absolute_error

mlp_predictions = model.predict(X)

lstm_mae = mean_absolute_error(y, data['lstm_forecast'])
sarima_mae = mean_absolute_error(y, data['sarima_forecast'])
mlp_mae = mean_absolute_error(y, mlp_predictions)

print(f'MAE for LSTM forecast: {lstm_mae:.2f}')
print(f'MAE for SARIMA forecast: {sarima_mae:.2f}')
print(f'MAE for MLP predictions: {mlp_mae:.2f}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
MAE for LSTM forecast: 90.09
MAE for SARIMA forecast: 96.56
MAE for MLP predictions: 83.80


In [ ]:
data['mlp_predictions'] = mlp_predictions
data

,date,152,lstm_forecast,sarima_forecast,Unnamed: 4,Unnamed: 5,mlp_predictions
0,01-01-2018 03:00,796,773.686584,820.627100,NaN,NaN,802.029297
1,01-01-2018 04:00,746,749.334168,718.173882,NaN,NaN,744.872498
2,01-01-2018 05:00,762,727.288696,690.631164,NaN,NaN,720.292175
3,01-01-2018 06:00,802,774.834534,748.946800,NaN,NaN,772.852722
4,01-01-2018 07:00,863,856.281311,847.535625,NaN,NaN,862.345337
5,01-01-2018 08:00,870,933.284241,904.162003,NaN,NaN,931.716003
6,01-01-2018 09:00,913,977.201782,915.990411,NaN,NaN,962.754578
7,01-01-2018 10:00,1064,1120.973633,1065.967452,NaN,NaN,1110.709229
8,01-01-2018 11:00,1379,1257.217041,1124.009391,NaN,NaN,1215.891235
9,01-01-2018 12:00,1635,1389.403564,1452.134065,NaN,NaN,1431.166626
